In [ ]:
import struct #Struct module in python is used to read binary data, which is what the .raw data is
import numpy as np
import matplotlib.pyplot as plt
import os

print('Libraries loaded successfully.')

Libraries loaded successfully.


In [6]:
def read_raw(file_path):
    #Accordining to the Clarius github page, the header has these: uint32 id, uint32 numFrames, 
    #uint32 numScanLines, uint32 numSamplesPerLine, uint32 sampleSizeInBytes
    #That would be 32*5/8 = 20 bytes in each header 

    #This uses python's struct module: https://docs.python.org/3/library/struct.html
    #Computers store binary as signed and unsigned data. Since RF data cycles from negative to positive,
    #signed data must be used. 

    header_format = '<5I' #5 uint32 = unsigned, integer, 32 bits of storage 
    #little-endian = bytes are stored least-significant first 
    #unsigned number because headers are positive numbers
    header_size = struct.calcsize(header_format) #Should be 20 bytes
    timestamp_format = '<Q' #5 uint64 = unsigned, integer, 62 bits, little endian 
    timestamp_size = struct.calcsize(timestamp_format) #1 byte = 8 bits, so we should have 8 bytes 


with open(file_path,'rb') as f: #Read the code in binary mode and save as my file = f
    raw_header = f.read(header_size)
    id_, num_frames, num_lines, num_samples, sample_size = struct.unpack(header_format, raw_header) 

    header = {
         'id': id_,
            'numFrames': num_frames,
            'numScanLines': num_lines,
            'numSamplesPerLine': num_samples,
            'sampleSizeInBytes': sample_size,
    }

    # RF data is always int16 (2 bytes per sample)
    assert sample_size == 2, f'Expected 2-byte samples for RF data, got {sample_size}'

    frame_data_bytes = num_lines * num_samples * sample_size

    timestamps = np.zeros(num_frames, dtype=np.uint64)
    rf_data = np.zeros((num_frames, num_lines, num_samples), dtype=np.int16)

    for i in range(num_frames):
        # --- Timestamp ---
        ts_raw = f.read(timestamp_size)
        timestamps[i] = struct.unpack(timestamp_format, ts_raw)[0]

        # --- RF frame data ---
        frame_raw = f.read(frame_data_bytes)
        frame_arr = np.frombuffer(frame_raw, dtype=np.int16)
        rf_data[i] = frame_arr.reshape(num_lines, num_samples)

return header, timestamps, rf_data

NameError: name 'file_path' is not defined